# YOLO11s Fine-Tuning

Training notebook for merged Carla Traffic + Carla f6 dataset. The final
label space has 12 classes.


## Colab Setup


In [ ]:
from pathlib import Path
import os
import sys

candidates = [
    Path("/content/fine_tuning"),
    Path.cwd(),
    Path.cwd() / "fine_tuning",
]

PROJECT_DIR = None
for candidate in candidates:
    if (candidate / "src" / "prepare_carla_dataset.py").exists():
        PROJECT_DIR = candidate.resolve()
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find the fine_tuning folder."
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

print(f"Project directory: {PROJECT_DIR}")


## Dependencies


In [ ]:
!pip -q install -r requirements_colab.txt


## Run Settings

Set the epoch count, and Roboflow key before preparing data or
training.


In [ ]:
import json
import subprocess
import shutil
import zipfile
from getpass import getpass

import pandas as pd
import torch

from common import load_yaml

CONFIG_PATH = "configs/roboflow_carla_traffic_f6_yolo11s.yaml"
config = load_yaml(CONFIG_PATH)
dataset_cfg = config["dataset"]

custom_dataset_dir = Path(dataset_cfg["custom_dataset_dir"])
baseline_dataset_dir = Path(dataset_cfg["baseline_dataset_dir"])
custom_data_yaml = custom_dataset_dir / "data.yaml"
baseline_data_yaml = baseline_dataset_dir / "data.yaml"
internal_dir = Path(config["outputs"].get("internal_dir", "workspace/internal"))
internal_dir.mkdir(parents=True, exist_ok=True)

EPOCHS = 5
RUN_TAG = "merged_5ep_smoke"

baseline_json = internal_dir / f"{RUN_TAG}_baseline.json"
train_json = internal_dir / f"{RUN_TAG}_train.json"
finetuned_json = internal_dir / f"{RUN_TAG}_finetuned.json"

if not os.environ.get("ROBOFLOW_API_KEY"):
    os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("cpu.")

print(f"Run tag: {RUN_TAG}")
print(f"Epochs: {EPOCHS}")
config


## Prepare Dataset

Build the custom 12-class view and the COCO-mapped baseline view from the
downloaded Roboflow datasets.


In [ ]:
cmd = [
    sys.executable,
    "src/prepare_carla_dataset.py",
    "--config",
    CONFIG_PATH,
    "--force",
]
subprocess.run(cmd, check=True)


## Dataset Summary

Check class order, split sizes, and the mapped unknown traffic-light/sign
classes before training.


In [ ]:
summary_path = Path(config["outputs"]["reports_dir"]) / "dataset_summary.json"
dataset_summary = json.loads(summary_path.read_text())

print("Custom class order:")
for class_id, name in dataset_summary["custom_names"].items():
    print(f"{class_id}: {name}")

split_rows = []
for split, info in dataset_summary["splits"].items():
    row = {
        "split": split,
        "images": info["images"],
        "empty_label_files": info["empty_label_files"],
    }
    row.update(info["objects_by_class_name"])
    split_rows.append(row)

display(pd.DataFrame(split_rows).fillna(0))

source_rows = []
for source_name, source_info in dataset_summary["source_datasets"].items():
    for source_split, split_info in source_info["splits"].items():
        row = {
            "source": source_name,
            "source_split": source_split,
            "target_split": split_info["target_split"],
            "images": split_info["images"],
            "empty_label_files": split_info["empty_label_files"],
        }
        row.update(split_info["objects_by_class_name"])
        source_rows.append(row)

display(pd.DataFrame(source_rows).fillna(0))


## Baseline Evaluation

Evaluate pretrained YOLO11s on the COCO-mapped dataset view for a fair baseline.


In [ ]:
baseline_classes = ",".join(str(x) for x in config["evaluation"]["baseline_class_filter"])
baseline_run_name = f"baseline_yolo11s_coco_mapped_{RUN_TAG}"

cmd = [
    sys.executable,
    "src/evaluate_yolo.py",
    "--model",
    config["training"]["base_model"],
    "--data",
    str(baseline_data_yaml),
    "--split",
    config["evaluation"]["split"],
    "--imgsz",
    str(config["training"]["image_size"]),
    "--device",
    DEVICE,
    "--project",
    config["outputs"]["runs_dir"],
    "--name",
    baseline_run_name,
    "--classes",
    baseline_classes,
    "--conf",
    str(config["evaluation"]["confidence"]),
    "--iou",
    str(config["evaluation"]["iou"]),
    "--label-space",
    "baseline_coco80",
    "--output-json",
    str(baseline_json),
]
subprocess.run(cmd, check=True)


## Training

Fine-tune YOLO11s on the custom CARLA label space.


In [ ]:

train_run_name = f"finetune_yolo11s_custom12_{RUN_TAG}"
print(f"training for {EPOCHS} epochs, batch={config['training']['batch']}, device={DEVICE}")

cmd = [
    sys.executable,
    "src/train_yolo.py",
    "--model",
    config["training"]["base_model"],
    "--data",
    str(custom_data_yaml),
    "--epochs",
    str(EPOCHS),
    "--imgsz",
    str(config["training"]["image_size"]),
    "--batch",
    str(config["training"]["batch"]),
    "--device",
    DEVICE,
    "--project",
    config["outputs"]["runs_dir"],
    "--name",
    train_run_name,
    "--patience",
    str(config["training"]["patience"]),
    "--seed",
    str(config["training"]["seed"]),
    "--workers",
    str(config["training"]["workers"]),
    "--optimizer",
    config["training"]["optimizer"],
    "--output-json",
    str(train_json),
]
subprocess.run(cmd, check=True)


## Fine-Tuned Evaluation

Evaluate the trained `best.pt` on the same validation split.


In [ ]:
train_summary = json.loads(train_json.read_text())
best_model = Path(train_summary["best_model"])

if not best_model.exists():
    raise FileNotFoundError(f"Expected best model was not found: {best_model}")

finetuned_run_name = f"finetuned_yolo11s_custom12_{RUN_TAG}_{config['evaluation']['split']}"

cmd = [
    sys.executable,
    "src/evaluate_yolo.py",
    "--model",
    str(best_model),
    "--data",
    str(custom_data_yaml),
    "--split",
    config["evaluation"]["split"],
    "--imgsz",
    str(config["training"]["image_size"]),
    "--device",
    DEVICE,
    "--project",
    config["outputs"]["runs_dir"],
    "--name",
    finetuned_run_name,
    "--conf",
    str(config["evaluation"]["confidence"]),
    "--iou",
    str(config["evaluation"]["iou"]),
    "--label-space",
    "custom12",
    "--output-json",
    str(finetuned_json),
]
subprocess.run(cmd, check=True)


## Metrics

Compare baseline and fine-tuned metrics, then review per-class AP for weak
classes.


In [ ]:
cmd = [
    sys.executable,
    "src/compare_metrics.py",
    "--config",
    CONFIG_PATH,
    "--baseline-json",
    str(baseline_json),
    "--finetuned-json",
    str(finetuned_json),
]
subprocess.run(cmd, check=True)

display(pd.read_csv(Path(config["outputs"]["reports_dir"]) / "metrics_comparison.csv"))
display(pd.read_csv(Path(config["outputs"]["reports_dir"]) / "per_class_ap50_comparison.csv"))


## Sample Predictions

Generate a small visual sample from the baseline and fine-tuned models.


In [ ]:
sample_images = custom_dataset_dir / "images" / config["evaluation"]["split"]
prediction_project = Path("workspace/runs/predictions")

cmd = [
    sys.executable,
    "src/predict_samples.py",
    "--model",
    config["training"]["base_model"],
    "--source-dir",
    str(sample_images),
    "--count",
    "12",
    "--seed",
    str(config["training"]["seed"]),
    "--imgsz",
    str(config["training"]["image_size"]),
    "--device",
    DEVICE,
    "--project",
    str(prediction_project),
    "--name",
    f"baseline_yolo11s_samples_{RUN_TAG}",
]
subprocess.run(cmd, check=True)

cmd = [
    sys.executable,
    "src/predict_samples.py",
    "--model",
    str(best_model),
    "--source-dir",
    str(sample_images),
    "--count",
    "12",
    "--seed",
    str(config["training"]["seed"]),
    "--imgsz",
    str(config["training"]["image_size"]),
    "--device",
    DEVICE,
    "--project",
    str(prediction_project),
    "--name",
    f"finetuned_yolo11s_samples_{RUN_TAG}",
]
subprocess.run(cmd, check=True)


## Export

Copy the best model and archive the reports and sample prediction outputs.


In [ ]:
reports_dir = Path(config["outputs"]["reports_dir"])
exports_dir = Path(config["outputs"]["exports_dir"])
exports_dir.mkdir(parents=True, exist_ok=True)

exported_best = exports_dir / f"yolo11s_carla_traffic_f6_{RUN_TAG}_best.pt"
shutil.copy2(best_model, exported_best)

zip_path = exports_dir / f"carla_yolo11s_{RUN_TAG}_results.zip"
include_paths = [
    reports_dir,
    Path("workspace/runs/predictions"),
    exported_best,
]

def archive_name(path):
    resolved = Path(path).resolve()
    return resolved.relative_to(PROJECT_DIR)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for item in include_paths:
        item = Path(item)
        if not item.exists():
            continue
        if item.is_file():
            zip_file.write(item, archive_name(item))
            continue
        for file_path in item.rglob("*"):
            if file_path.is_file():
                zip_file.write(file_path, archive_name(file_path))

print(f"best model: {exported_best}")
print(f"archive: {zip_path}")
